In [1]:
import backtrader as bt
import datetime
import pandas as pd


In [2]:
from mongofeed import MongoData

td = datetime.datetime.combine(datetime.date.today(),datetime.time())
# 创业etf
feed_1 = MongoData(
    db='stock_etf',
    dataname='etf_159915',
    fromdate=datetime.datetime(2013,1,1),
    todate=td, 
)
# 恒生
feed_2 = MongoData(
    db='stock_etf',
    dataname='etf_159920',
    fromdate=datetime.datetime(2013,1,1),
    todate=td, 
)

In [3]:
class KDJ(bt.Indicator):
    lines = ('K','D','J')
    params = (('period', 9),('slowk', 3),('slowd',3))

    plotinfo = dict(
        # Add extra margins above and below the 1s and -1s
        #plotymargin=0.01,

        # Plot a reference horizontal line
        plotyhlines=[10, 90],

        # Simplify the y scale to 1.0 and -1.0
        #plotyticks=[0.0, 6.0])
    )
    
    plotlines = dict(
#         op=dict( marker='o', color='purple',ls='',
#                   markersize=5.0, fillstyle='full'),
        
        #K=dict(_fill_gt=('D', ('orange', 0.5)), _fill_lt=('D', ('green', 0.5))),
        #D=dict(color='black')
        #J=dict(ls='--', _fill_gt=('K', ('red', 0.5)), _fill_lt=('K', ('blue', 0.5))),
    )

    def __init__(self):
        self.kd = bt.indicators.StochasticFull(
            self.data,
            period=self.p.period,
            period_dfast=self.p.slowk,
            period_dslow=self.p.slowd,
        )
        self.l.K = self.kd.percD
        self.l.D = self.kd.percDSlow
        self.l.J = self.l.K*3 - self.l.D*2
        #self.cross = bt.indicators.CrossOver(self.l.K, self.l.D)
        
    def next(self):
#         if self.l.J<0.0:
#             self.l.op[0] = self.l.J-30 
#         elif self.l.J>100.0:
#             self.l.op[0] = self.l.J+30
        pass

In [4]:
# Create a Stratey
class EmptyStrategy(bt.Strategy):
    '''
    Empty strategy is used to study indicators
    '''
    params = (
        ('none', 0),
    )
    
    def log(self, txt):
        ''' Logging function for this strategy'''
        dt = self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))
        
    def __init__(self):
        # ex indicators
#         bt.talib.EMA(self.data0.close, timeperiod=20)
#         bt.talib.SMA(self.data0.close, timeperiod=20)
#         bt.talib.KAMA(self.data0.close, timeperiod=20)
        
        #self.mfi = bt.talib.MFI(self.data0.high, self.data0.low, self.data0.close, self.data0.volume)
        
        # cycle
        #self.ht_dc = bt.talib.HT_DCPERIOD(self.data0)
        
        self.ind0 = bt.indicators.RSI_Safe(self.data0)
        self.ind1 = bt.indicators.RSI_Safe(self.data1)
        
    def next(self):
        #self.log(self.data0.datetime.date())
        #self.log(self.data1.datetime.date())
        wd = self.data0.datetime.date().weekday()
        if wd == 0:
            # monday
            rd = self.data1 if self.ind0>self.ind1 else self.data0
            self.buy(data=rd)
            self.log('buy')
        
        if wd == 4:
            # friday
            posdict = self.getpositions()
            
            rd = self.data0 if posdict[self.data0] else self.data1
            self.close(data=rd)
            self.log('close')
    
    def stop(self):
        pass

In [5]:
cerebro = bt.Cerebro() # oldtrades=False, stdstats=False

cerebro.adddata(feed_1, name='cy_daily')
cerebro.adddata(feed_2, name='hsi_daily')
#cerebro.resampledata(feed, name='cy_weekly', timeframe=bt.TimeFrame.Weeks)

cerebro.addstrategy(EmptyStrategy)

# exp indicators
#cerebro.addindicator(bt.talib.AROON,feed.high,feed.low)

# 小场面1万起始资金
cerebro.broker.setcash(10000.0)

# 手续费万5
cerebro.broker.setcommission(0.0005)

cerebro.broker.set_coc(True)
cerebro.broker.set_fundstartval(50)

cerebro.addsizer(bt.sizers.AllInSizerInt, percents=99)

print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())

result = cerebro.run()

print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

Starting Portfolio Value: 10000.00
2013-01-24, 2013-01-24
2013-01-24, 2013-01-24
2013-01-25, 2013-01-25
2013-01-25, 2013-01-25
2013-01-25, close
2013-01-28, 2013-01-28
2013-01-28, 2013-01-28
2013-01-28, buy
2013-01-29, 2013-01-29
2013-01-29, 2013-01-29
2013-01-30, 2013-01-30
2013-01-30, 2013-01-30
2013-01-31, 2013-01-31
2013-01-31, 2013-01-31
2013-02-01, 2013-02-01
2013-02-01, 2013-02-01
2013-02-01, close
2013-02-04, 2013-02-04
2013-02-04, 2013-02-04
2013-02-04, buy
2013-02-05, 2013-02-05
2013-02-05, 2013-02-05
2013-02-06, 2013-02-06
2013-02-06, 2013-02-06
2013-02-07, 2013-02-07
2013-02-07, 2013-02-07
2013-02-08, 2013-02-08
2013-02-08, 2013-02-08
2013-02-08, close
2013-02-18, 2013-02-18
2013-02-18, 2013-02-18
2013-02-18, buy
2013-02-19, 2013-02-19
2013-02-19, 2013-02-19
2013-02-20, 2013-02-20
2013-02-20, 2013-02-20
2013-02-21, 2013-02-21
2013-02-21, 2013-02-21
2013-02-22, 2013-02-22
2013-02-22, 2013-02-22
2013-02-22, close
2013-02-25, 2013-02-25
2013-02-25, 2013-02-25
2013-02-25, buy
2

2013-12-23, 2013-12-23
2013-12-23, 2013-12-23
2013-12-23, buy
2013-12-24, 2013-12-24
2013-12-24, 2013-12-24
2013-12-25, 2013-12-25
2013-12-25, 2013-12-25
2013-12-26, 2013-12-26
2013-12-26, 2013-12-26
2013-12-27, 2013-12-27
2013-12-27, 2013-12-27
2013-12-27, close
2013-12-30, 2013-12-30
2013-12-30, 2013-12-30
2013-12-30, buy
2013-12-31, 2013-12-31
2013-12-31, 2013-12-31
2014-01-02, 2014-01-02
2014-01-02, 2014-01-02
2014-01-03, 2014-01-03
2014-01-03, 2014-01-03
2014-01-03, close
2014-01-06, 2014-01-06
2014-01-06, 2014-01-06
2014-01-06, buy
2014-01-07, 2014-01-07
2014-01-07, 2014-01-07
2014-01-08, 2014-01-08
2014-01-08, 2014-01-08
2014-01-09, 2014-01-09
2014-01-09, 2014-01-09
2014-01-10, 2014-01-10
2014-01-10, 2014-01-10
2014-01-10, close
2014-01-13, 2014-01-13
2014-01-13, 2014-01-13
2014-01-13, buy
2014-01-14, 2014-01-14
2014-01-14, 2014-01-14
2014-01-15, 2014-01-15
2014-01-15, 2014-01-15
2014-01-16, 2014-01-16
2014-01-16, 2014-01-16
2014-01-17, 2014-01-17
2014-01-17, 2014-01-17
2014-01-

2014-09-30, 2014-09-30
2014-09-30, 2014-09-30
2014-10-08, 2014-10-08
2014-10-08, 2014-10-08
2014-10-09, 2014-10-09
2014-10-09, 2014-10-09
2014-10-10, 2014-10-10
2014-10-10, 2014-10-10
2014-10-10, close
2014-10-13, 2014-10-13
2014-10-13, 2014-10-13
2014-10-13, buy
2014-10-14, 2014-10-14
2014-10-14, 2014-10-14
2014-10-15, 2014-10-15
2014-10-15, 2014-10-15
2014-10-16, 2014-10-16
2014-10-16, 2014-10-16
2014-10-17, 2014-10-17
2014-10-17, 2014-10-17
2014-10-17, close
2014-10-20, 2014-10-20
2014-10-20, 2014-10-20
2014-10-20, buy
2014-10-21, 2014-10-21
2014-10-21, 2014-10-21
2014-10-22, 2014-10-22
2014-10-22, 2014-10-22
2014-10-23, 2014-10-23
2014-10-23, 2014-10-23
2014-10-24, 2014-10-24
2014-10-24, 2014-10-24
2014-10-24, close
2014-10-27, 2014-10-27
2014-10-27, 2014-10-27
2014-10-27, buy
2014-10-28, 2014-10-28
2014-10-28, 2014-10-28
2014-10-29, 2014-10-29
2014-10-29, 2014-10-29
2014-10-30, 2014-10-30
2014-10-30, 2014-10-30
2014-10-31, 2014-10-31
2014-10-31, 2014-10-31
2014-10-31, close
2014-1

2015-09-22, 2015-09-22
2015-09-23, 2015-09-23
2015-09-23, 2015-09-23
2015-09-24, 2015-09-24
2015-09-24, 2015-09-24
2015-09-25, 2015-09-25
2015-09-25, 2015-09-25
2015-09-25, close
2015-09-28, 2015-09-28
2015-09-28, 2015-09-28
2015-09-28, buy
2015-09-29, 2015-09-29
2015-09-29, 2015-09-29
2015-09-30, 2015-09-30
2015-09-30, 2015-09-30
2015-10-08, 2015-10-08
2015-10-08, 2015-10-08
2015-10-09, 2015-10-09
2015-10-09, 2015-10-09
2015-10-09, close
2015-10-12, 2015-10-12
2015-10-12, 2015-10-12
2015-10-12, buy
2015-10-13, 2015-10-13
2015-10-13, 2015-10-13
2015-10-14, 2015-10-14
2015-10-14, 2015-10-14
2015-10-15, 2015-10-15
2015-10-15, 2015-10-15
2015-10-16, 2015-10-16
2015-10-16, 2015-10-16
2015-10-16, close
2015-10-19, 2015-10-19
2015-10-19, 2015-10-19
2015-10-19, buy
2015-10-20, 2015-10-20
2015-10-20, 2015-10-20
2015-10-21, 2015-10-21
2015-10-21, 2015-10-21
2015-10-22, 2015-10-22
2015-10-22, 2015-10-22
2015-10-23, 2015-10-23
2015-10-23, 2015-10-23
2015-10-23, close
2015-10-26, 2015-10-26
2015-1

2016-05-30, buy
2016-05-31, 2016-05-31
2016-05-31, 2016-05-31
2016-06-01, 2016-06-01
2016-06-01, 2016-06-01
2016-06-02, 2016-06-02
2016-06-02, 2016-06-02
2016-06-03, 2016-06-03
2016-06-03, 2016-06-03
2016-06-03, close
2016-06-06, 2016-06-06
2016-06-06, 2016-06-06
2016-06-06, buy
2016-06-07, 2016-06-07
2016-06-07, 2016-06-07
2016-06-08, 2016-06-08
2016-06-08, 2016-06-08
2016-06-13, 2016-06-13
2016-06-13, 2016-06-13
2016-06-13, buy
2016-06-14, 2016-06-14
2016-06-14, 2016-06-14
2016-06-15, 2016-06-15
2016-06-15, 2016-06-15
2016-06-16, 2016-06-16
2016-06-16, 2016-06-16
2016-06-17, 2016-06-17
2016-06-17, 2016-06-17
2016-06-17, close
2016-06-20, 2016-06-20
2016-06-20, 2016-06-20
2016-06-20, buy
2016-06-21, 2016-06-21
2016-06-21, 2016-06-21
2016-06-22, 2016-06-22
2016-06-22, 2016-06-22
2016-06-23, 2016-06-23
2016-06-23, 2016-06-23
2016-06-24, 2016-06-24
2016-06-24, 2016-06-24
2016-06-24, close
2016-06-27, 2016-06-27
2016-06-27, 2016-06-27
2016-06-27, buy
2016-06-28, 2016-06-28
2016-06-28, 201

2017-02-21, 2017-02-21
2017-02-21, 2017-02-21
2017-02-22, 2017-02-22
2017-02-22, 2017-02-22
2017-02-23, 2017-02-23
2017-02-23, 2017-02-23
2017-02-24, 2017-02-24
2017-02-24, 2017-02-24
2017-02-24, close
2017-02-27, 2017-02-27
2017-02-27, 2017-02-27
2017-02-27, buy
2017-02-28, 2017-02-28
2017-02-28, 2017-02-28
2017-03-01, 2017-03-01
2017-03-01, 2017-03-01
2017-03-02, 2017-03-02
2017-03-02, 2017-03-02
2017-03-03, 2017-03-03
2017-03-03, 2017-03-03
2017-03-03, close
2017-03-06, 2017-03-06
2017-03-06, 2017-03-06
2017-03-06, buy
2017-03-07, 2017-03-07
2017-03-07, 2017-03-07
2017-03-08, 2017-03-08
2017-03-08, 2017-03-08
2017-03-09, 2017-03-09
2017-03-09, 2017-03-09
2017-03-10, 2017-03-10
2017-03-10, 2017-03-10
2017-03-10, close
2017-03-13, 2017-03-13
2017-03-13, 2017-03-13
2017-03-13, buy
2017-03-14, 2017-03-14
2017-03-14, 2017-03-14
2017-03-15, 2017-03-15
2017-03-15, 2017-03-15
2017-03-16, 2017-03-16
2017-03-16, 2017-03-16
2017-03-17, 2017-03-17
2017-03-17, 2017-03-17
2017-03-17, close
2017-0

2017-11-03, 2017-11-03
2017-11-03, 2017-11-03
2017-11-03, close
2017-11-06, 2017-11-06
2017-11-06, 2017-11-06
2017-11-06, buy
2017-11-07, 2017-11-07
2017-11-07, 2017-11-07
2017-11-08, 2017-11-08
2017-11-08, 2017-11-08
2017-11-09, 2017-11-09
2017-11-09, 2017-11-09
2017-11-10, 2017-11-10
2017-11-10, 2017-11-10
2017-11-10, close
2017-11-13, 2017-11-13
2017-11-13, 2017-11-13
2017-11-13, buy
2017-11-14, 2017-11-14
2017-11-14, 2017-11-14
2017-11-15, 2017-11-15
2017-11-15, 2017-11-15
2017-11-16, 2017-11-16
2017-11-16, 2017-11-16
2017-11-17, 2017-11-17
2017-11-17, 2017-11-17
2017-11-17, close
2017-11-20, 2017-11-20
2017-11-20, 2017-11-20
2017-11-20, buy
2017-11-21, 2017-11-21
2017-11-21, 2017-11-21
2017-11-22, 2017-11-22
2017-11-22, 2017-11-22
2017-11-23, 2017-11-23
2017-11-23, 2017-11-23
2017-11-24, 2017-11-24
2017-11-24, 2017-11-24
2017-11-24, close
2017-11-27, 2017-11-27
2017-11-27, 2017-11-27
2017-11-27, buy
2017-11-28, 2017-11-28
2017-11-28, 2017-11-28
2017-11-29, 2017-11-29
2017-11-29, 2

2018-07-18, 2018-07-18
2018-07-18, 2018-07-18
2018-07-19, 2018-07-19
2018-07-19, 2018-07-19
2018-07-20, 2018-07-20
2018-07-20, 2018-07-20
2018-07-20, close
2018-07-23, 2018-07-23
2018-07-23, 2018-07-23
2018-07-23, buy
2018-07-24, 2018-07-24
2018-07-24, 2018-07-24
2018-07-25, 2018-07-25
2018-07-25, 2018-07-25
2018-07-26, 2018-07-26
2018-07-26, 2018-07-26
2018-07-27, 2018-07-27
2018-07-27, 2018-07-27
2018-07-27, close
2018-07-30, 2018-07-30
2018-07-30, 2018-07-30
2018-07-30, buy
2018-07-31, 2018-07-31
2018-07-31, 2018-07-31
2018-08-01, 2018-08-01
2018-08-01, 2018-08-01
2018-08-02, 2018-08-02
2018-08-02, 2018-08-02
2018-08-03, 2018-08-03
2018-08-03, 2018-08-03
2018-08-03, close
2018-08-06, 2018-08-06
2018-08-06, 2018-08-06
2018-08-06, buy
2018-08-07, 2018-08-07
2018-08-07, 2018-08-07
2018-08-08, 2018-08-08
2018-08-08, 2018-08-08
2018-08-09, 2018-08-09
2018-08-09, 2018-08-09
2018-08-10, 2018-08-10
2018-08-10, 2018-08-10
2018-08-10, close
2018-08-13, 2018-08-13
2018-08-13, 2018-08-13
2018-0

2019-08-23, 2019-08-23
2019-08-23, 2019-08-23
2019-08-23, close
2019-08-26, 2019-08-26
2019-08-26, 2019-08-26
2019-08-26, buy
2019-08-27, 2019-08-27
2019-08-27, 2019-08-27
2019-08-28, 2019-08-28
2019-08-28, 2019-08-28
2019-08-29, 2019-08-29
2019-08-29, 2019-08-29
2019-08-30, 2019-08-30
2019-08-30, 2019-08-30
2019-08-30, close
2019-09-02, 2019-09-02
2019-09-02, 2019-09-02
2019-09-02, buy
2019-09-03, 2019-09-03
2019-09-03, 2019-09-03
2019-09-04, 2019-09-04
2019-09-04, 2019-09-04
2019-09-05, 2019-09-05
2019-09-05, 2019-09-05
2019-09-06, 2019-09-06
2019-09-06, 2019-09-06
2019-09-06, close
2019-09-09, 2019-09-09
2019-09-09, 2019-09-09
2019-09-09, buy
2019-09-10, 2019-09-10
2019-09-10, 2019-09-10
2019-09-11, 2019-09-11
2019-09-11, 2019-09-11
2019-09-12, 2019-09-12
2019-09-12, 2019-09-12
2019-09-16, 2019-09-16
2019-09-16, 2019-09-16
2019-09-16, buy
2019-09-17, 2019-09-17
2019-09-17, 2019-09-17
2019-09-18, 2019-09-18
2019-09-18, 2019-09-18
2019-09-19, 2019-09-19
2019-09-19, 2019-09-19
2019-09-

In [6]:
params = dict(
    style='candle',
    barup='#FF0033',
    bardown='#32CD32',
    volup='#F66269',
    voldown='#43A047',
)
cerebro.plot(
    iplot=False, 
    **params
)

[[<Figure size 1366x681 with 8 Axes>]]